In [52]:
import torch
import torch.optim as optim
from torchvision import datasets
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset , DataLoader
from network import Network , Loader
import pickle
import torch.nn as nn
from torch.utils.data import Dataset , DataLoader
import torch


In [53]:
data = datasets.MNIST('./Data',train = True , download = True)

In [54]:
X = data.data
y = data.targets

X_train , X_test , y_train , y_test = train_test_split(X,y,test_size = 0.2,shuffle = True)
X_train =  X_train

X_train = X_train.float()/255

Y_train = y_train.long()

x_test = X_test.float()
x_test = x_test/255

y_test = y_test.long()



In [55]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

device


device(type='cpu')

In [56]:
Train_loader = Loader(X_train,y_train)
Train_loader = DataLoader(Train_loader,batch_size = 32,shuffle = True)


test_loader = Loader(x_test,y_test)
test_loader = DataLoader(test_loader,batch_size = 32,shuffle = False)

In [ ]:

class Loader(Dataset):
  def __init__(self,x,y):
    self.x = torch.tensor(x,dtype = torch.float32).reshape(-1,1,28,28)
    self.y = torch.tensor(y,dtype = torch.long)
  def __len__(self):
    return len(self.x)
  def __getitem__(self,item):
    return self.x[item],self.y[item]


In [58]:

model = Network(1).to(device)
optimizer = optim.Adam(model.parameters(),lr = 0.001)
criteria = torch.nn.CrossEntropyLoss()

In [59]:
epoch = 20
for e in range(epoch):

  total_loss = 0

  for X,y in Train_loader:

    X = X.to(device)
    y = y.to(device)

    X = X.reshape(-1,1,28,28)
    y_pred = model(X)

    loss = criteria(y_pred,y)

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    total_loss += loss.item()
  print(f"epoch = {e+1} , loss = {total_loss/len(Train_loader)}")



epoch = 1 , loss = 0.23048810161751074
epoch = 2 , loss = 0.09451522165373899
epoch = 3 , loss = 0.0749829529215834
epoch = 4 , loss = 0.06394773806914843
epoch = 5 , loss = 0.052760392351601695
epoch = 6 , loss = 0.04734715412032044
epoch = 7 , loss = 0.043725419024561916
epoch = 8 , loss = 0.03819256579148381
epoch = 9 , loss = 0.03794311477863512
epoch = 10 , loss = 0.0386157734418295
epoch = 11 , loss = 0.03236873573787952
epoch = 12 , loss = 0.03084539254754413
epoch = 13 , loss = 0.028002496029271632
epoch = 14 , loss = 0.028919004020950437
epoch = 15 , loss = 0.02839600771989538
epoch = 16 , loss = 0.025935305917625256
epoch = 17 , loss = 0.024610002086684138
epoch = 18 , loss = 0.024162858363468896
epoch = 19 , loss = 0.022806618125096306
epoch = 20 , loss = 0.024012871440432734


In [ ]:
count = 0
total = 0
model.eval()
with torch.no_grad():
  for item , label in test_loader:

    item = item.to(device)

    label = label.to(device)

    y_out = model(item)

    predictions = torch.argmax(y_out, dim=1)
    total += label.shape[0]

    count += (predictions == label).sum().item()

print("correct --> ", count, " accuracy -->", (count / total) * 100, "%")

correct -->  11877  accuracy --> 98.97500000000001 %


In [ ]:
print(torch.argmax(y_out,dim=1))
print(y_test)


tensor([4, 5, 1,  ..., 7, 2, 6])
tensor([4, 5, 1,  ..., 7, 2, 6])


In [ ]:
model = model.to('cpu')
torch.save(model,'/content/modelv2')

In [ ]:
from torchvision import transforms
from PIL import Image

input = Image.open("to_predict/image.png").convert("L")
transform = transforms.ToTensor()
input = transform(input)
print(input.shape)
input = input.reshape(-1,1,28,28).to(device)

output = model(input)

result = torch.argmax(output,dim = 1)

print(result)